# WHO Gateway Sitemap Scraper with Local Storage

This notebook demonstrates how to:
1. Download and save a webpage (WHO Gateway sitemap) locally as HTML
2. Parse the local HTML file with BeautifulSoup
3. Extract and save data to a SQLite database
4. Save the BeautifulSoup object for future use

This approach helps avoid 403 Forbidden errors by downloading the content once and then working with the local copy.

## Workflow:
1. Download the sitemap once and save it as HTML
2. Parse the local HTML file with BeautifulSoup
3. Extract data from the parsed content
4. Save the BeautifulSoup object for future use

## Benefits:
- Avoids repeated network requests that might trigger 403 errors
- Speeds up development by working with local files
- Preserves the original HTML for reference and debugging

In [11]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import re
import time
from urllib.parse import urljoin
import random
import os

In [4]:
# Create SQLite database
conn = sqlite3.connect('ind.db')
cursor = conn.cursor()

# Create tables
cursor.execute('''
CREATE TABLE IF NOT EXISTS categories (
    id INTEGER PRIMARY KEY,
    name TEXT,
    url TEXT UNIQUE,
    parent_id INTEGER
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS indicators (
    id INTEGER PRIMARY KEY,
    name TEXT,
    url TEXT UNIQUE,
    category_id INTEGER,
    description TEXT
)
''')

In [8]:
# Function to extract data from the sitemap
def extract_from_sitemap(sitemap_url):
   # Add proper headers to mimic a browser request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Referer': 'https://gateway.euro.who.int/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Cache-Control': 'max-age=0'
    }
    
    print(f"Attempting to access: {sitemap_url}")
    try:
        response = requests.get(sitemap_url)
        if response.status_code != 200:
            print(f"Failed to retrieve sitemap: {response.status_code}")
            return
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract all links from the sitemap
        links = soup.find_all('a')
        
        # First pass: extract categories
        for link in links:
            url = urljoin(sitemap_url, link.get('href'))
            name = link.text.strip()
            
            # Skip empty or non-health indicator links
            if not name or url == sitemap_url:
                continue
                
            # Determine if this is a category or indicator
            is_category = any(keyword in url for keyword in 
                              ['/indicators/', '/data-stories/','/how-to/','/datasets/','/country-profiles/'])
            
            if is_category:
                # Check if category already exists
                cursor.execute("SELECT id FROM categories WHERE url = ?", (url,))
                result = cursor.fetchone()
                if not result:
                    cursor.execute("INSERT INTO categories (name, url) VALUES (?, ?)", (name, url))
                    conn.commit()
                    
        # Second pass: extract indicators and their relationships
        for link in links:
            url = urljoin(sitemap_url, link.get('href'))
            name = link.text.strip()
            
            # Skip empty links
            if not name or url == sitemap_url:
                continue
                
            # Check if this looks like an indicator page
            is_indicator = '/indicator/' in url or '/indicators/' in url
            
            if is_indicator:
                # Find the best matching category
                cursor.execute("SELECT id, url FROM categories")
                categories = cursor.fetchall()
                best_category_id = None
                longest_match = 0
                
                for cat_id, cat_url in categories:
                    if url.startswith(cat_url) and len(cat_url) > longest_match:
                        best_category_id = cat_id
                        longest_match = len(cat_url)
                
                # Insert indicator
                try:
                    cursor.execute("INSERT INTO indicators (name, url, category_id) VALUES (?, ?, ?)", 
                                   (name, url, best_category_id))
                    conn.commit()
                    print(f"Added indicator: {name}")
                except sqlite3.IntegrityError:
                    # Skip duplicates
                    pass
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")


In [10]:
# Main execution
sitemap_url = "https://gateway.euro.who.int/en/htmlsitemap/"
extract_from_sitemap(sitemap_url)

# Optionally fetch more details about each indicator
#fetch_indicator_descriptions()

# Print summary
#cursor.execute("SELECT COUNT(*) FROM categories")
#category_count = cursor.fetchone()[0]

#cursor.execute("SELECT COUNT(*) FROM indicators")
#indicator_count = cursor.fetchone()[0]

#print(f"Extracted {category_count} categories and {indicator_count} indicators")

# Close connection
conn.close()

Attempting to access: https://gateway.euro.who.int/en/htmlsitemap/
Failed to retrieve sitemap: 403
Failed to retrieve sitemap: 403


In [12]:
# Function to download and save the sitemap HTML locally
def download_and_save_sitemap(url, save_path="sitemap.html"):
    """
    Downloads the sitemap from the given URL and saves it locally
    
    Args:
        url: URL of the sitemap
        save_path: Local file path to save the HTML
        
    Returns:
        True if successful, False otherwise
    """
    print(f"Attempting to download sitemap from: {url}")
    
    # Create browser-like headers to avoid 403 errors
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://gateway.euro.who.int/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Cache-Control': 'max-age=0',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none'
    }
    
    try:
        # First try with complete headers
        response = requests.get(url, headers=headers, timeout=30)
        
        # If we still get 403, try with a different approach
        if response.status_code == 403:
            print("Got 403 with first attempt, trying with a simpler approach...")
            
            # Create a session to maintain cookies
            session = requests.Session()
            
            # First access the main site to get cookies
            main_url = "https://gateway.euro.who.int/en/"
            session.get(main_url, headers=headers)
            
            # Then try to access the sitemap
            response = session.get(url, headers=headers, timeout=30)
        
        # Check the response
        if response.status_code == 200:
            # Save the HTML content to a file
            with open(save_path, 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            print(f"Successfully downloaded and saved sitemap to: {save_path}")
            return True
        else:
            print(f"Failed to download sitemap. Status code: {response.status_code}")
            print(f"Response headers: {response.headers}")
            return False
            
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return False
    except Exception as e:
        print(f"Unexpected error: {e}")
        return False

# Try to download the sitemap
sitemap_url = "https://gateway.euro.who.int/en/htmlsitemap/"
sitemap_file = "who_sitemap.html"
download_success = download_and_save_sitemap(sitemap_url, sitemap_file)

Attempting to download sitemap from: https://gateway.euro.who.int/en/htmlsitemap/
Successfully downloaded and saved sitemap to: who_sitemap.html


In [ ]:
# Function to load the local HTML file and parse with BeautifulSoup
def parse_local_sitemap(html_file):
    """
    Loads a local HTML file and parses it with BeautifulSoup
    
    Args:
        html_file: Path to the local HTML file
        
    Returns:
        BeautifulSoup object or None if failed
    """
    if not os.path.exists(html_file):
        print(f"File not found: {html_file}")
        return None
        
    try:
        print(f"Loading HTML file: {html_file}")
        
        # Open and read the HTML file
        with open(html_file, 'r', encoding='utf-8') as f:
            html_content = f.read()
            
        # Parse with BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')
        print(f"Successfully parsed {html_file} with BeautifulSoup")
        return soup
        
    except Exception as e:
        print(f"Error parsing HTML file: {e}")
        return None
        
# Check if we successfully downloaded the file, then parse it
if download_success and os.path.exists(sitemap_file):
    sitemap_soup = parse_local_sitemap(sitemap_file)
    
    # Display some basic stats about the sitemap
    if sitemap_soup:
        links = sitemap_soup.find_all('a')
        print(f"Found {len(links)} links in the sitemap")
        
        # Show the first few links as a sample
        print("\nSample of links found:")
        for i, link in enumerate(links[:5]):
            href = link.get('href')
            text = link.text.strip()
            print(f"{i+1}. {text} - {href}")
else:
    print("Sitemap not downloaded or file not found.")

In [ ]:
# Modified function to extract data from local sitemap file
def extract_from_local_sitemap(soup, sitemap_base_url):
    """
    Extract data from a local BeautifulSoup object representing the sitemap
    
    Args:
        soup: BeautifulSoup object of the parsed sitemap
        sitemap_base_url: Base URL to use for resolving relative links
    """
    if not soup:
        print("No soup object provided")
        return
        
    print(f"Extracting data from local sitemap using base URL: {sitemap_base_url}")
    
    # Connect to database
    conn = sqlite3.connect('ind.db')
    cursor = conn.cursor()
    
    try:
        # Extract all links from the sitemap
        links = soup.find_all('a')
        
        # First pass: extract categories
        for link in links:
            href = link.get('href')
            if not href:
                continue
                
            url = urljoin(sitemap_base_url, href)
            name = link.text.strip()
            
            # Skip empty or non-health indicator links
            if not name or url == sitemap_base_url:
                continue
                
            # Determine if this is a category or indicator
            is_category = any(keyword in url for keyword in 
                              ['/indicators/', '/data-stories/','/how-to/','/datasets/','/country-profiles/'])
            
            if is_category:
                # Check if category already exists
                cursor.execute("SELECT id FROM categories WHERE url = ?", (url,))
                result = cursor.fetchone()
                if not result:
                    cursor.execute("INSERT INTO categories (name, url) VALUES (?, ?)", (name, url))
                    conn.commit()
                    print(f"Added category: {name}")
                    
        # Second pass: extract indicators and their relationships
        for link in links:
            href = link.get('href')
            if not href:
                continue
                
            url = urljoin(sitemap_base_url, href)
            name = link.text.strip()
            
            # Skip empty links
            if not name or url == sitemap_base_url:
                continue
                
            # Check if this looks like an indicator page
            is_indicator = '/indicator/' in url or '/indicators/' in url
            
            if is_indicator:
                # Find the best matching category
                cursor.execute("SELECT id, url FROM categories")
                categories = cursor.fetchall()
                best_category_id = None
                longest_match = 0
                
                for cat_id, cat_url in categories:
                    if url.startswith(cat_url) and len(cat_url) > longest_match:
                        best_category_id = cat_id
                        longest_match = len(cat_url)
                
                # Insert indicator
                try:
                    cursor.execute("INSERT INTO indicators (name, url, category_id) VALUES (?, ?, ?)", 
                                   (name, url, best_category_id))
                    conn.commit()
                    print(f"Added indicator: {name}")
                except sqlite3.IntegrityError:
                    # Skip duplicates
                    pass
    except Exception as e:
        print(f"Extraction error: {e}")
    finally:
        # Close database connection
        conn.close()

# If we have the soup object, try to extract data
if 'sitemap_soup' in locals() and sitemap_soup:
    # Use the same base URL as the original sitemap
    extract_from_local_sitemap(sitemap_soup, sitemap_url)

In [ ]:
# Functions to save and load BeautifulSoup objects as pickled files
import pickle
from pathlib import Path

def save_soup_to_pickle(soup, file_path="sitemap_soup.pickle"):
    """
    Save BeautifulSoup object as a pickle file
    
    Args:
        soup: BeautifulSoup object to save
        file_path: Path to save the pickle file
    """
    try:
        with open(file_path, 'wb') as f:
            pickle.dump(soup, f)
        print(f"Saved soup object to {file_path}")
        return True
    except Exception as e:
        print(f"Error saving soup object: {e}")
        return False

def load_soup_from_pickle(file_path="sitemap_soup.pickle"):
    """
    Load BeautifulSoup object from a pickle file
    
    Args:
        file_path: Path to the pickle file
        
    Returns:
        BeautifulSoup object or None if failed
    """
    if not os.path.exists(file_path):
        print(f"Pickle file not found: {file_path}")
        return None
        
    try:
        with open(file_path, 'rb') as f:
            soup = pickle.load(f)
        print(f"Loaded soup object from {file_path}")
        return soup
    except Exception as e:
        print(f"Error loading soup object: {e}")
        return None

# Save the soup object for future use if we have it
if 'sitemap_soup' in locals() and sitemap_soup:
    save_soup_to_pickle(sitemap_soup)

# Example of how to load the soup object in a future session
# soup = load_soup_from_pickle()

In [ ]:
# Query and visualize the database
def show_database_stats():
    """Display database statistics"""
    # Connect to database
    conn = sqlite3.connect('ind.db')
    cursor = conn.cursor()
    
    try:
        # Count categories
        cursor.execute("SELECT COUNT(*) FROM categories")
        category_count = cursor.fetchone()[0]
        
        # Count indicators
        cursor.execute("SELECT COUNT(*) FROM indicators")
        indicator_count = cursor.fetchone()[0]
        
        print(f"Database Statistics:")
        print(f"- {category_count} categories")
        print(f"- {indicator_count} indicators")
        
        # Show top categories with most indicators
        cursor.execute("""
            SELECT c.name, COUNT(i.id) as indicator_count
            FROM categories c
            JOIN indicators i ON c.id = i.category_id
            GROUP BY c.id
            ORDER BY indicator_count DESC
            LIMIT 5
        """)
        
        print("\nTop Categories:")
        for name, count in cursor.fetchall():
            print(f"- {name}: {count} indicators")
            
        # Show some sample indicators
        cursor.execute("""
            SELECT i.name, c.name as category
            FROM indicators i
            LEFT JOIN categories c ON i.category_id = c.id
            ORDER BY RANDOM()
            LIMIT 5
        """)
        
        print("\nSample Indicators:")
        for name, category in cursor.fetchall():
            print(f"- {name} (Category: {category or 'Unknown'})")
            
    except Exception as e:
        print(f"Error querying database: {e}")
    finally:
        conn.close()

# Run the stats function to see what we've collected
show_database_stats()